# Лабораторная работа 03. Сводный анализ данных в Pandas

**Курс:** «Анализ данных и искусственный интеллект», 1 курс бакалавриата.

**Задача аналитика:** изучить предложения автомобилей на вторичном рынке и подготовить основу для оценки цены.

После работы вы сможете рассчитывать описательные статистики, сравнивать группы, строить сводные таблицы и простые графики средствами Pandas.

**План:** структура данных → статистики → `groupby` → `pivot_table` и доли → простая оценка цены → `Pandas.plot`.
Расширенная визуализация и бизнес-представление результатов вынесены в лабораторную 04.

Основные разделы рассчитаны на одну практику; проверка оценки цены на новых данных — дополнительное задание.


Разведочный анализ данных (EDA) помогает понять набор данных до построения модели: какие объекты представлены, какие значения типичны и какие записи требуют проверки. В этой работе основной инструмент — таблица, а простые графики помогают проверить выводы.

Одна строка описывает автомобиль в наборе предложений. `Price` — указанная цена в **евро**, `Year` — **год выпуска**, `Distance` — пробег в километрах. В данных нет даты продажи, факта сделки или времени нахождения объявления на сайте. Поэтому число записей характеризует представленность в выборке, а не спрос; указанная цена не обязательно равна цене сделки.


## Что должен выяснить аналитик

1. Какие марки и сегменты хорошо представлены в выборке?
2. Какая цена типична для каждого сегмента и насколько велик разброс?
3. Где данных мало и выводы требуют осторожности?
4. Может ли средняя или медианная цена похожих автомобилей служить простой оценкой?

Редкое значение не обязательно является ошибкой. Запись сначала проверяют по нескольким признакам, затем обосновывают решение о её обработке.


## 1. Загрузка данных


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

# Положите CSV рядом с ноутбуком. Для примера из репозитория — резервный адрес.
DATA_FILE = 'cars_processed.csv'
data_path = Path(DATA_FILE)
if data_path.exists():
    df = pd.read_csv(data_path)
else:
    df = pd.read_csv(
        'https://raw.githubusercontent.com/MVRonkin/BasicDataAnalysisCourse/'
        'refs/heads/main/Workshops/cars_processed.csv'
    )

# Совместимость с результатом предыдущей практики.
df = df.rename(columns={'Price(euro)': 'Price', 'Engine_capacity(cm3)': 'Engine'})
df = df.drop(columns=[c for c in df.columns if c.startswith('Unnamed:')])
if 'Transmission' not in df:
    df['Transmission'] = df['AutoTransmission'].map({True: 'Automatic', False: 'Manual'})
if 'km_year_cat' not in df:
    df['km_year_cat'] = pd.cut(
        df['km_year'], bins=[-np.inf, 10_000, 30_000, np.inf],
        labels=['Small', 'Medium', 'Large']
    )
# Номер строки нужен для возврата от графика к исходной записи; это не ID объявления.
df['row_id'] = np.arange(len(df))
num_columns = ['Year', 'Distance', 'Engine', 'Price', 'km_year']
cat_columns = ['Make', 'Model', 'Style', 'Fuel_type', 'Transmission', 'km_year_cat']
df.head()


### Входной файл


Используйте `cars_processed.csv` из лабораторной 02 или свой файл с теми же колонками. `cars_cleaned.csv` — более поздний результат очистки; для воспроизведения примеров он не нужен. Исходный CSV не перезаписывается.


## 2. Первичная проверка

Проверим размер таблицы, типы колонок, пропуски и точные дубликаты. До удаления дубликатов запишем их число. Одинаковые описания разных объявлений могут совпадать; здесь удаление точных повторов используется как учебное допущение.


In [ ]:
df.info()

In [ ]:
df.shape

2. Вспомним про то, как проверить данные на количество пропусков и дубликатов

In [ ]:
df.isna().sum().sum() # Пропуски

In [ ]:
df.duplicated().sum()

In [ ]:
duplicates_count = int(df.drop(columns='row_id').duplicated().sum())
print('Точных повторов описаний:', duplicates_count)
df = df.drop_duplicates(subset=[c for c in df.columns if c != 'row_id']).copy()


## 3. Сводный анализ признаков


Во многих случаях аналитики должны предоставить агрегированные или сгруппированные данные. Эта информация помогает понять общие характеристики в определенных разрезах данных. Часто этого достаточно, чтобы достичь бизнес-цели или принимать решения.

Также сводный анализ позволяет достаточно просто нати наиболее редкие записи данных или наиболее редкие комбинации данные

### Описательные статистики

Попробуем проанализировать данные 

1. посмотрим на то, какие у нас есть категориальные и численные столбцы.
   
[Напомним, что категориальные переменные](https://education.yandex.ru/handbook/data-analysis/article/rabota-s-kategorialnymi-dannymi-v-python) — это те переменные, которые выражены ограниченным диапазоном значений. Например, группа крови или семейное положение.



In [ ]:
display(pd.DataFrame({'Численные': pd.Series(num_columns),
                      'Категориальные': pd.Series(cat_columns)}))


Статистика по числовым данным

In [ ]:
df.describe()

Прочитайте `count`, `mean`, `std`, `min`, `25%`, `50%`, `75%`, `max`.
`50%` — медиана; между `25%` и `75%` лежит центральная половина наблюдений.
Среднее чувствительно к крайним значениям. Максимум сам по себе не доказывает ошибку.

**Вопросы:** насколько различаются средняя и медианная цена? Какие максимальные значения нужно проверить? Почему квартиль года выпуска не является основанием удалять более старые автомобили?


Аналогично примеру выше посчитаем статистик по категориальным признакам. Это будут признаки с указанием типов данных `object` и `bool`

In [ ]:
df[cat_columns].astype('object').describe()


Другим важным инструментом анализа является группировка данных. 

В общем случае группировка данных в Pandas выглядит следующим образом:
```python
df.groupby(by=grouping_columns)[columns_to_show].function()
```

1. К датафрейму применяется метод `groupby`, который групперует данные по `grouping_columns` – признаку или набору признаков.
2. Выбираем нужные нам столбцы (`columns_to_show`).
3. К полученным группам применяется функция или несколько функций.

Попробуем сгруппировать данные по производителю ('Make'). Посмотрим сколько автомобилей кажого производителя имеется по годам. В качестве функции группировки будем использовать подсчет значений в группе (`count`), минимальный и максимальный год автомобиля в группе `np.min, np.max`. Для того, чтобы использовать сразу 3 функции вместе воспользуемся методом `agg`

In [ ]:
df.groupby(['Make','Year'])[['Price']].mean()

### Упражнение

Выведите 10 марок с самой высокой средней ценой. Затем проверьте, сколько автомобилей приходится на каждую марку: рейтинг по одной дорогой записи может быть нестабильным.


In [ ]:
df.groupby('Make')['Price'].mean().sort_values(ascending=False).head(10)

### Усиление 1. Среднее, медиана и размер группы

`agg` рассчитывает несколько показателей за один вызов. `size` считает строки, `count` — известные значения выбранной колонки. Именованная агрегация позволяет дать результатам понятные названия.

Порог `30` ниже выбран для учебного отчёта; он не гарантирует статистическую надёжность.


In [ ]:
make_summary = df.groupby('Make').agg(
    n=('Price', 'size'), known_price=('Price', 'count'),
    mean_price=('Price', 'mean'), median_price=('Price', 'median'),
    q1=('Price', lambda s: s.quantile(0.25)),
    q3=('Price', lambda s: s.quantile(0.75))
)
make_summary['share_pct'] = 100 * make_summary['n'] / len(df)
make_summary['iqr'] = make_summary['q3'] - make_summary['q1']
display(make_summary.sort_values('n', ascending=False).head(10).round(1))
display(make_summary.query('n >= 30').nlargest(10, 'median_price').round(1))


**Задание:** сравните рейтинг по средней цене и по медиане. Найдите марку, для которой вывод изменился. Укажите размер группы и объясните расхождение. Ни один рейтинг не доказывает, что марка выгоднее для перепродажи: затрат и цен сделок в данных нет.


## 4. Построение сводных таблиц


Анализ при помощи сводных таблиц называется __pivot_table__

В качестве примера анализ __PIVOT__ рассмотрим задачу оценки популярность типов кузова по маркам автомобилей.

* `data=df` - Исходные данные
* `index='Make'` - Группировка по марке — будет в строках
* `columns='Style'` - Группировка по типу кузова — будет в столбцах
* `aggfunc='size'` - Функция агрегации: считает количество строк в каждой группе
* `fill_value=0` - Заменяет отсутствующие значения (`NaN`) на `0`

In [ ]:
pivot_style = pd.pivot_table(df, index=['Make','Year'], columns=['Style'] , values = 'Price',  aggfunc='size', fill_value=0)
# pivot_style = pd.pivot_table(df, index=['Make'], columns=['Style'] ,  aggfunc='size', fill_value=0)
pivot_style

In [ ]:
pivot_style = pd.pivot_table(
    df, index=['Make','Year'], columns=['Style'],
    aggfunc='size', fill_value=0
)

# Сумма по каждому кузову
style_totals = pivot_style.sum(axis=0).sort_values(ascending=False)
print(style_totals)

# Самый популярный
top_style = style_totals.idxmax()
print("Самый популярный кузов:", top_style)

К результатам pivot можно обращаться через методы `loc` и `iloc`

Попробуем теперь посмотреть сводку по конкретным маркам 

In [ ]:
pivot_style.loc[['Toyota', 'BMW']]

Определим сколько у 'Toyota' записей 'Sedan'

In [ ]:
pivot_style.loc['Toyota', 'Sedan']

Посчитаем общие количество кузовов кажого типа

In [ ]:
pivot_style.sum(axis=0)

Или отсортеруем сводку по тому у какой марке больше всего седанов

In [ ]:
pivot_style.sort_values(by='Sedan',ascending = False)

Сводка может производится по нескольким параметрам. Например с целью определить аномалии проверим в каких марках и по каким годам распределены кузова

In [ ]:
pivot_style = pd.pivot_table(df, index=['Make',], columns='Style', aggfunc='size', fill_value=0)
pivot_style

In [ ]:
pivot_style = pd.pivot_table(df, index=['Make',], columns='Year', aggfunc='size', fill_value=0)
pivot_style

Отметим, что аналогичных результатов `PIVOT` можно получить несколькими способами:
1. `groupby()` - более гибкий, позволяет добавлять дополнительные операции или делать более детальную группировку, но это менее читаемый способ. 
```python
groupby_style = df.groupby(['Make', 'Year', 'Style']).size().unstack(fill_value=0) #groupby c использованием unstack
```

2. `crosstab()` - [простой, но не гибкий сопособ подсчета значений](https://sky.pro/wiki/analytics/analiz-dannyh-s-pdcrosstab-v-python-sozdaem-svodnye-tablicy)
```python
crosstab_style = pd.crosstab(index=[df['Make'], df['Year']], columns=df['Style'])
```

3. `pivot_table()` - способ агрегации по значениям, а не просто подсчет значений
```python
pivot_style = pd.pivot_table(df, index=['Make','Year'], columns='Style', aggfunc='size', fill_value=0)
```
   

### Усиление 2. Число записей и доля — разные вопросы

`crosstab` считает сочетания категорий. `normalize='index'` делит каждую строку на её сумму: получаем структуру кузовов **внутри марки**.

В таблице цен отсутствие наблюдений оставляем как `NaN`. Ноль означал бы известную нулевую цену.


In [ ]:
style_count = pd.crosstab(df['Make'], df['Style'])
style_share = 100 * pd.crosstab(df['Make'], df['Style'], normalize='index')
top_makes = df['Make'].value_counts().head(10).index
display(style_count.loc[top_makes])
display(style_share.loc[top_makes].round(1))
price_pivot = df.pivot_table(index='Make', columns='Transmission',
                             values='Price', aggfunc='median')
display(price_pivot.loc[top_makes].round(0))


**Задание:** найдите марку с наибольшим числом SUV и марку с наибольшей долей SUV среди марок с хотя бы 30 записями. Совпадают ли ответы? Укажите знаменатель каждой доли.


### Упражнения: редкие комбинации

1. Найдите редкие сочетания марки и кузова. Для трёх примеров изучите исходные записи и предложите решение: оставить, уточнить у источника или исправить подтверждённую ошибку.
2. Обсудите объединение `Roadster` и `Cabriolet` в аналитическую группу. Это изменение детализации отчёта, а не исправление ошибки. Сохраните исходный `Style`.

Чистота данных помогает анализу, но равное число объектов в группах не гарантирует лучшую модель. Объединение категорий требует предметного обоснования.


In [ ]:
df['Style_group'] = df['Style'].replace(
    {'Roadster': 'Cabriolet/Roadster', 'Cabriolet': 'Cabriolet/Roadster'}
)


In [ ]:
pd.crosstab(df['Make'], df['Style_group'])


3. Предложите заполнение пропусков в численной колонке медианой похожих автомобилей. Выберите признаки группировки и правило для группы, где нет известных значений. В упражнении скрывайте только известные значения входного признака в копии таблицы, чтобы оценить ошибку заполнения.

Не заменяйте дорогой автомобиль средней ценой только из-за редкости. `Price` — целевая переменная: её искусственное заполнение создаёт выдуманные ответы для будущей модели.


4. Для каждой марки определите самый частый кузов. Обсудите заполнение **пропущенного**, а не просто редкого кузова модой группы. Как поступить при равных частотах и неизвестной марке?


5. Постройте таблицу числа автомобилей по маркам и десятилетиям **выпуска**. Используйте `(Year // 10) * 10`. Это распределение по возрастным когортам; для `resample` нужен временной индекс, которого здесь нет.


In [ ]:
df_numeric_and_make = df[num_columns + ['Make']].copy()
df_numeric_and_make['Decade'] = (df_numeric_and_make['Year'] // 10) * 10
pivot_make = pd.pivot_table(df_numeric_and_make, index='Make',
                            columns='Decade', aggfunc='size', fill_value=0)
pivot_make.sum()


## 5. Простая оценка цены по группе

Средняя цена автомобилей той же марки и года выпуска — простой ориентир. Он не учитывает модель, комплектацию и состояние. Построим такую таблицу и измерим разброс цен вокруг группового среднего.

$$\mathrm{MAE}=\frac{1}{N}\sum_{i=1}^{N}|y_i-\hat y_i|.$$

Здесь $y_i$ — указанная цена автомобиля, $\hat y_i$ — оценка. MAE выражается в евро.
На тех же данных, из которых рассчитаны средние, MAE описывает внутригрупповое отклонение; качество прогноза новых автомобилей требует отдельной проверки.


In [ ]:
df_numeric_and_make = df[list(num_columns) + ['Make']]
pivot_make = pd.pivot_table(df_numeric_and_make[['Make','Year','Price']], 
                           index=['Make'], 
                           columns='Year', 
                           values='Price',  # явно указываем колонку для агрегации
                           aggfunc='mean')  # используем строку вместо np.mean для наглядности

проверим как это работает для одного производителя в заданный год. Года будем выбирать в диапазоне от 2010 до 2020, из анализа выше видно, что это наиболее распростроненный период.

In [ ]:
make = 'Audi'
years = range(2010, 2020)
year = 2015
year


In [ ]:
price_mean = pivot_make.loc[make, year]
print(f"Средняя цена {make} в {year} году: {float(price_mean):.2f}")

In [ ]:
price_mean

In [ ]:
price_by_year = df[(df['Make'] == make) & (df['Year']==year)]['Price']
mae = (price_by_year - price_mean).abs().mean()
print(f"Cреднее абсолютное отклонение (MAE) для оценки по pivot автомобиля {make} в {year} году: {mae:.2f}")

Подсчитаем значение ошибки MAE за заданный период для заданного производителя. Для этого зададим фильтр данных и создадим в нем новый столбец `mean_by_year`

In [ ]:
# выбираем часть набора данных по производитель - диапазон year
filtered_df = df[(df['Make'] == make) & (df['Year'].isin(years))].copy()
#заполняем столбец в формате год - средняя цена
filtered_df['mean_by_year'] = filtered_df['Year'].map(pivot_make.loc[make]) 


mae = (filtered_df['Price']-filtered_df['mean_by_year']).abs().mean()

print(f'Cреднее абсолютное отклонение (MAE) оценки по pivot для автомобилей {make} {years[0]} - {years[-1]} года = {mae:.1f}  ')

Также на самом деле можно бы было подсчитать среднее MAE для всего набора данных

воспользуемся неочевидным вариантом функция `transform` применяет функцию к серии и возвращает массив исходной размерности. 

In [ ]:
mean_by_group = df.groupby(['Make', 'Year'])['Price'].transform('mean')
mae = (mean_by_group - df['Price']).abs().mean()
print(f'Внутригрупповое среднее абсолютное отклонение: {mae:.1f} евро')


Среднее группы содержит цену самого оцениваемого автомобиля. У группы из одной записи отклонение равно нулю — это не свидетельство точного прогноза.

**Упражнения:** сравните среднее и медиану группы; добавьте `Model`; объясните, почему дробление групп снижает отклонение на исходных данных, но может ухудшить оценку новых автомобилей.

Дополнительно рассчитайте MAPE для положительных цен. Низкие цены имеют большой вес в процентной ошибке, поэтому приводите MAPE вместе с MAE.


## 6. Простые графики Pandas


### От сводной таблицы к графику


`Series.plot` и `DataFrame.plot` позволяют быстро показать результат группировки. Отдельно изучать API Matplotlib в этой работе не требуется, но библиотека Matplotlib должна быть установлена.

Покажем среднюю цену Mercedes по **годам выпуска**. Линия связывает разные группы автомобилей в одной выборке; она не показывает изменение рыночной цены во времени.


In [ ]:
make = 'Mercedes'
filter_df = df[df['Make'] ==make]
filter_df_by_year = filter_df.groupby('Year')['Price'].mean()

In [ ]:
filter_df_by_year.plot();


Обратите внимание, что результат groupby имеет индексты - года!

In [ ]:
filter_df_by_year.index

Сделаем более приятный для чтения график

In [ ]:
filter_df_by_year.plot(
    marker='o', figsize=(12, 3),
    title=f'Средняя цена {make} по годам выпуска',
    xlabel='Год выпуска', ylabel='Средняя цена, евро',
    color='tab:blue', grid=True
);


Тот же результат можно представить столбчатой диаграммой. Сравните читаемость при большом числе годов. Выберите вариант, который лучше отвечает на ваш вопрос.


In [ ]:
filter_df_by_year.plot(
    kind='bar', figsize=(12, 3),
    title=f'Средняя цена {make} по годам выпуска',
    xlabel='Год выпуска', ylabel='Средняя цена, евро', color='tab:blue'
);


## 7. Самостоятельная работа и результат

Подготовьте небольшой табличный отчёт:

1. Пять показателей набора: число записей, медиана цены, квартильный диапазон цены, медиана пробега, число марок.
2. Сравнение не менее трёх сегментов с ценой, разбросом и числом записей.
3. Две сводные таблицы: абсолютные количества и доли. Объясните различие выводов.
4. Два графика `Pandas.plot` с единицами измерения и подписью, объясняющей результат.
5. Три вывода: наблюдение в числах → возможное объяснение → ограничение вывода.

**Что сдавать:** выполненный ноутбук. Не включайте среднюю цену группы, рассчитанную по всей таблице, во входные признаки будущей модели: она использует целевую переменную.

Сводные таблицы показали состав выборки и типичные цены сегментов. Следующий шаг — визуально проверить распределения, сравнить сегменты и представить результаты для заказчика в лабораторной 04.


### Передача данных в лабораторную 04

Сохраняем исходные признаки и учебные категории без оценок, рассчитанных из цены. Новый файл не заменяет CSV из лабораторной 02.


In [ ]:
export_columns = [c for c in df.columns if c != 'mean_make_price_by_year']
df[export_columns].to_csv('cars_summary_input.csv', index=False)
make_summary.to_csv('cars_make_summary.csv', index=True)


## Дополнительно. Проверка простой оценки на отложенных данных

Это задание можно выполнить после основного отчёта. Разделите строки случайно в отношении 80/20 с фиксированным seed. Таблицу средних стройте только на первой части. Для неизвестной группы используйте среднюю цену первой части и отдельно сообщайте долю таких случаев. Сравните MAE с общей средней ценой.

Так проверяется перенос на другие строки той же выборки. Для проверки будущего рынка потребовались бы даты объявлений; повторные объявления одного автомобиля также могут завысить результат случайного разделения.


In [ ]:
train = df.sample(frac=0.8, random_state=42)
test = df.drop(train.index).copy()
group_mean = train.groupby(['Make', 'Year'])['Price'].mean().rename('estimate')
test = test.join(group_mean, on=['Make', 'Year'])
unknown = test['estimate'].isna()
fallback = train['Price'].mean()
test['estimate'] = test['estimate'].fillna(fallback)
print(f'Новых групп: {unknown.mean():.1%}')
print(f'MAE общей средней: {(test["Price"] - fallback).abs().mean():.1f} евро')
print(f'MAE групповой оценки: {(test["Price"] - test["estimate"]).abs().mean():.1f} евро')


Документация: [группировка и агрегация Pandas](https://pandas.pydata.org/docs/user_guide/groupby.html).
